In [1]:
import logging
import os
from pathlib import Path
import duckdb
from google.colab import drive
import pandas as pd

# 1. CONECTAR O GOOGLE DRIVE
try:
  drive.mount('/content/drive')
except Exception:
  pass

# Configuração de Logs
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s'
)
log = logging.getLogger('gold_pipeline')

# 2. DEFINIÇÃO DOS CAMINHOS DO PROJETO NO DRIVE
BASE = Path(
    os.environ.get(
        'ESCORPIOES_BASE', '/content/drive/MyDrive/projeto_escorpioes'
    )
)
SINAN_SILVER = (
    BASE
    / 'sinan'
    / '2_silver'
    / 'sinan_cotia_regiao_escorpioes_2016_2026_silver.csv'
)
INMET_SILVER = BASE / 'inmet' / '2_silver' / 'inmet_diario_2016_2026.csv'
PASTA_GOLD = BASE / '3_gold'


def executar_pipeline_gold():
  log.info('🚀 Iniciando cruzamento para a Camada GOLD via DuckDB...')

  # Validação de existência dos arquivos Silver
  for arq in (SINAN_SILVER, INMET_SILVER):
    if not arq.exists():
      raise FileNotFoundError(f'❌ Arquivo Silver não encontrado: {arq}')

  PASTA_GOLD.mkdir(parents=True, exist_ok=True)

  # Conectar ao DuckDB em memória
  con = duckdb.connect()

  # A. Ler os CSVs Silver como Views temporárias no DuckDB
  con.execute(f"""
        CREATE OR REPLACE VIEW silver_sinan AS
        SELECT * FROM read_csv('{SINAN_SILVER}', delim=';', header=true, all_varchar=true);

        CREATE OR REPLACE VIEW silver_inmet AS
        SELECT * FROM read_csv('{INMET_SILVER}', delim=';', header=true, all_varchar=true);
    """)

  # B. SQL 1: Tabela GOLD por Notificação (Caso + Clima do dia do acidente)
  query_gold_casos = """
        CREATE OR REPLACE TABLE gold_casos_clima AS
        SELECT
            s.*,
            TRY_CAST(i.PRECIPITACAO_TOTAL_MM AS DOUBLE) AS "Precipitação Total (mm)",
            TRY_CAST(i.TEMPERATURA_MEDIA_C AS DOUBLE)   AS "Temperatura Média (°C)",
            TRY_CAST(i.UMIDADE_RELATIVA_MEDIA AS DOUBLE)  AS "Umidade Relativa Média (%)",
            CASE WHEN i.DATA IS NULL THEN 'Não' ELSE 'Sim' END AS "Clima Disponível"
        FROM silver_sinan s
        LEFT JOIN silver_inmet i
            ON TRY_CAST(s."Data do Acidente" AS DATE) = TRY_CAST(i.DATA AS DATE)
        ORDER BY s."Data do Acidente", s."Data da Notificação";
    """
  con.execute(query_gold_casos)

  # C. Exportação para CSV na pasta 3_gold
  df_gold_casos = con.execute('SELECT * FROM gold_casos_clima').df()
  destino_casos = PASTA_GOLD / 'gold_casos_clima_2016_2026.csv'
  df_gold_casos.to_csv(
      destino_casos, sep=';', index=False, encoding='utf-8-sig'
  )

  # D. Salva o arquivo .sql como documentação da query utilizada
  (PASTA_GOLD / 'gold_escorpioes_clima.sql').write_text(
      query_gold_casos, encoding='utf-8'
  )

  # Validações de Qualidade
  n_total = len(df_gold_casos)
  com_clima = len(df_gold_casos[df_gold_casos['Clima Disponível'] == 'Sim'])

  print('\n' + '=' * 70)
  print('🎉 CAMADA GOLD CONCLUÍDA COM SUCESSO!')
  print(f'📍 Arquivo consolidado salvo em: {destino_casos}')
  print(
      f'📊 Total de Casos: {n_total} | Casos com Clima do Dia Acoplado:'
      f' {com_clima} ({com_clima / n_total:.1%})'
  )
  print('=' * 70)

  con.close()


# Executar
if __name__ == '__main__':
  executar_pipeline_gold()

Mounted at /content/drive

🎉 CAMADA GOLD CONCLUÍDA COM SUCESSO!
📍 Arquivo consolidado salvo em: /content/drive/MyDrive/projeto_escorpioes/3_gold/gold_casos_clima_2016_2026.csv
📊 Total de Casos: 7181 | Casos com Clima do Dia Acoplado: 7176 (99.9%)
